# Training-data ingestion — brief §8.2

**EXPERIMENTAL.** Videos → clean TTS dataset (speech clips + transcripts) +
face crops for Phase 4. Your originals are never modified — path + sha1 only.

**Getting videos in without Drive:** cell 4 uploads them straight from your
computer. Colab's uploader is slow for big files, so **shrink them first** on
your machine (one command, cell 3 shows it) — a 50 MB clip → ~5 MB, still fine
for this. Or use cell 4c to pull from a direct link.

`Runtime → T4 GPU`, run top to bottom.


In [ ]:
#@title 1. GPU check
!nvidia-smi -L || print('NO GPU — Runtime > Change runtime type > T4 GPU')


In [ ]:
%%writefile /content/ingest.py
#!/usr/bin/env python3
"""
Training-data ingestion — brief §8.2. EXPERIMENTAL, GPU helps (Whisper).

Turns the founder's authorized videos into:
  dataset/wavs/*.wav          speech clips, 2.5–12 s, mono 22.05 kHz
  dataset/metadata.csv        LJSpeech-style  <id>|<text>
  dataset/metadata.jsonl      richer rows (duration, source, asr, quality)
  dataset/faces/*.jpg         sampled face crops (for the Phase 4 face profile)
  report.json                 per-video quality score + status

Model-agnostic: this output feeds whichever TTS we fine-tune AND the face
identity profile. Originals are never modified or deleted — only their paths
and sha1 are recorded (brief §8.2, §8.7).

Pipeline per video (brief §8.2):
  extract audio → split on silence → transcribe (Whisper) → align → quality
  score → (frames → detect face → face score) → add to dataset

Usage:
  python ingest.py --videos ./videos --out ./out [--whisper small] [--lang sw]
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import pathlib
import re
import subprocess
import sys

SR = 22050
MIN_SEC = 2.5
MAX_SEC = 12.0
VIDEO_EXT = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v", ".wav", ".m4a", ".mp3"}


def sh(cmd: list[str]) -> str:
    return subprocess.run(cmd, capture_output=True, text=True).stderr


def sha1(path: pathlib.Path) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def extract_audio(src: pathlib.Path, dst: pathlib.Path) -> None:
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-vn", "-ac", "1", "-ar", str(SR), str(dst)],
        check=True, capture_output=True,
    )


def duration(path: pathlib.Path) -> float:
    err = sh(["ffmpeg", "-i", str(path)])
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+\.\d+)", err)
    if not m:
        return 0.0
    return int(m[1]) * 3600 + int(m[2]) * 60 + float(m[3])


def adaptive_noise_db(mean_vol_db: float) -> float:
    """Silence threshold relative to the recording's own loudness, not a fixed
    number. A quiet phone-mic recording (mean around -30 to -35dB) has actual
    speech sitting close to or below a flat -30dB cutoff, so a fixed threshold
    misclassifies almost the whole file as "silence" and near-zero speech gets
    recognised. Set the cutoff clearly below the recording's average instead,
    clamped to a sane range."""
    return max(-45.0, min(-18.0, mean_vol_db - 10.0))


def silence_windows(wav: pathlib.Path, noise_db: float = -30, min_sil: float = 0.4):
    """Return (start, end) speech spans between detected silences.

    Natural speech is full of short pauses well under MIN_SEC on its own — a
    filter that drops any individual span shorter than MIN_SEC silently throws
    away most real conversational audio. Instead, short spans separated by a
    brief gap are MERGED into one clip (a pause inside a clip is normal); only
    a genuinely negligible leftover fragment (<0.6s) is dropped.
    """
    err = sh(["ffmpeg", "-i", str(wav), "-af",
              f"silencedetect=noise={noise_db}dB:d={min_sil}", "-f", "null", "-"])
    starts = [float(x) for x in re.findall(r"silence_start:\s*([\d.]+)", err)]
    ends = [float(x) for x in re.findall(r"silence_end:\s*([\d.]+)", err)]
    total = duration(wav)
    # speech = complement of the silence intervals
    sil = sorted(zip(starts, ends + [total] * (len(starts) - len(ends))))
    spans, cur = [], 0.0
    for s, e in sil:
        if s - cur > 0.3:
            spans.append((cur, s))
        cur = e
    if total - cur > 0.3:
        spans.append((cur, total))

    # merge spans forward across short gaps until each clip reaches MIN_SEC
    merged: list[tuple[float, float]] = []
    for s, e in spans:
        if merged and s - merged[-1][1] <= 1.2 and (merged[-1][1] - merged[-1][0]) < MIN_SEC:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))

    # enforce length bounds: hard-cut > MAX, drop only negligible leftovers
    out = []
    for s, e in merged:
        while e - s > MAX_SEC:
            out.append((s, s + MAX_SEC))
            s += MAX_SEC
        if e - s >= 0.6:
            out.append((s, e))
    return out


def cut(wav: pathlib.Path, s: float, e: float, dst: pathlib.Path) -> None:
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(wav), "-ss", f"{s:.3f}", "-to", f"{e:.3f}",
         "-ac", "1", "-ar", str(SR), str(dst)],
        check=True, capture_output=True,
    )


def mean_volume_db(wav: pathlib.Path) -> float:
    err = sh(["ffmpeg", "-i", str(wav), "-af", "volumedetect", "-f", "null", "-"])
    m = re.search(r"mean_volume:\s*(-?[\d.]+)\s*dB", err)
    return float(m[1]) if m else -99.0


# --------------------------------------------------------------------------- #

_whisper = None


def transcribe(wav: pathlib.Path, model_size: str, lang: str) -> tuple[str, float]:
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel  # type: ignore
        import torch  # noqa

        dev = "cuda" if _cuda() else "cpu"
        _whisper = WhisperModel(model_size, device=dev,
                                compute_type="float16" if dev == "cuda" else "int8")
    # vad_filter=False: we already pre-segment on silence with our own
    # loudness-adaptive detector (adaptive_noise_db) before a clip ever reaches
    # here. Whisper's *own* internal VAD (Silero, fixed sensitivity) applied on
    # top of that was a second, uncalibrated filter silently discarding real
    # speech in quiet recordings — it doesn't know this file's loudness.
    segs, _info = _whisper.transcribe(str(wav), language=lang, vad_filter=False)
    parts, logp = [], []
    for s in segs:
        parts.append(s.text.strip())
        logp.append(getattr(s, "avg_logprob", 0.0))
    text = re.sub(r"\s+", " ", " ".join(parts)).strip()
    conf = float(sum(logp) / len(logp)) if logp else -9.0
    return text, conf


def _cuda() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except Exception:
        return False


# --------------------------------------------------------------------------- #

_facedet = None


def face_scores(video: pathlib.Path, out_dir: pathlib.Path, n: int = 12):
    """Sample n frames, return (found_ratio, mean_face_frac, mean_sharpness, saved).

    Uses OpenCV's Haar cascade (bundled with opencv-python, a stable decades-old
    API) rather than mediapipe's legacy `mp.solutions` detector, which recent
    mediapipe releases removed outright ('module mediapipe has no attribute
    solutions') — this sidesteps that version fragility entirely."""
    global _facedet
    try:
        import cv2  # type: ignore
    except Exception as exc:  # noqa: BLE001
        return {"error": f"face libs unavailable: {exc}"}

    if _facedet is None:
        _facedet = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    cap = cv2.VideoCapture(str(video))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    if total <= 0:
        return {"error": "no frames"}
    idxs = [int(total * (i + 0.5) / n) for i in range(n)]
    found, fracs, sharps, saved = 0, [], [], 0
    for k, fi in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = _facedet.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
        if len(faces) == 0:
            continue
        x, y, cw, ch = max(faces, key=lambda f: f[2] * f[3])
        found += 1
        fracs.append(max(0.0, min(1.0, ch / h)))
        crop = frame[y:y + ch, x:x + cw]
        if crop.size:
            sharps.append(cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY),
                                        cv2.CV_64F).var())
            if k < 6:
                cv2.imwrite(str(out_dir / f"{video.stem}_{k}.jpg"), crop)
                saved += 1
    cap.release()
    return {
        "found_ratio": round(found / max(1, len(idxs)), 2),
        "mean_face_frac": round(sum(fracs) / len(fracs), 3) if fracs else 0.0,
        "mean_sharpness": round(sum(sharps) / len(sharps), 1) if sharps else 0.0,
        "faces_saved": saved,
    }


# --------------------------------------------------------------------------- #

def classify(speech_sec: float, n_clips: int, vol_db: float, face: dict) -> tuple[int, str]:
    score = 10
    notes = []
    if vol_db < -34:
        score -= 3
        notes.append("TOO MUCH BACKGROUND NOISE / low level")
    if speech_sec < 60:
        score -= 3
        notes.append("NEEDS MORE SPEECH (<60s usable)")
    if n_clips < 8:
        score -= 1
    ff = face.get("found_ratio", 0.0)
    frac = face.get("mean_face_frac", 0.0)
    if "error" not in face:
        if ff < 0.5 or frac < 0.12:
            score -= 3
            notes.append("FACE NOT CLEAR ENOUGH")
        if face.get("mean_sharpness", 0) and face["mean_sharpness"] < 40:
            score -= 2
            notes.append("FACE BLURRY")
    score = max(1, score)
    status = "GOOD FOR TRAINING" if score >= 7 and not notes else (
        "; ".join(notes) if notes else "USABLE")
    return score, status


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--videos", required=True)
    ap.add_argument("--out", default="./out")
    ap.add_argument("--whisper", default="medium")  # "small" under-transcribes Swahili
    ap.add_argument("--lang", default="sw")
    ap.add_argument("--photos", default="", help="optional dir of face photos "
                    "(jpg/png) — used when you upload audio instead of video")
    a = ap.parse_args()

    vroot = pathlib.Path(a.videos)
    out = pathlib.Path(a.out)
    ds = out / "dataset"
    (ds / "wavs").mkdir(parents=True, exist_ok=True)
    (ds / "faces").mkdir(parents=True, exist_ok=True)
    (out / "work").mkdir(parents=True, exist_ok=True)

    photos_n = 0
    if a.photos and pathlib.Path(a.photos).is_dir():
        import shutil as _sh

        for p in sorted(pathlib.Path(a.photos).iterdir()):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                _sh.copy(p, ds / "faces" / p.name)
                photos_n += 1
        print(f"copied {photos_n} face photos -> dataset/faces/")

    vids = sorted(p for p in vroot.iterdir() if p.suffix.lower() in VIDEO_EXT)
    if not vids:
        sys.exit(f"no media in {vroot}")

    meta_csv = open(ds / "metadata.csv", "w", newline="", encoding="utf-8")
    meta_jsonl = open(ds / "metadata.jsonl", "w", encoding="utf-8")
    w = csv.writer(meta_csv, delimiter="|")
    report = []
    total_clips = 0

    for v in vids:
        print(f"\n=== {v.name} ===")
        full_wav = out / "work" / f"{v.stem}.wav"
        extract_audio(v, full_wav)
        vol_db = mean_volume_db(full_wav)
        spans = silence_windows(full_wav, noise_db=adaptive_noise_db(vol_db))
        print(f"  {len(spans)} candidate clips, mean volume {vol_db:.1f} dB")

        kept, speech_sec = 0, 0.0
        for i, (s, e) in enumerate(spans):
            clip_id = f"{v.stem}_{i:04d}"
            clip = ds / "wavs" / f"{clip_id}.wav"
            cut(full_wav, s, e, clip)
            text, conf = transcribe(clip, a.whisper, a.lang)
            if len(text) < 3 or conf < -2.2:
                clip.unlink(missing_ok=True)
                continue
            w.writerow([f"wavs/{clip_id}.wav", text])
            meta_jsonl.write(json.dumps({
                "id": clip_id, "wav": f"wavs/{clip_id}.wav", "text": text,
                "dur": round(e - s, 2), "asr_conf": round(conf, 3),
                "source": v.name,
            }, ensure_ascii=False) + "\n")
            kept += 1
            speech_sec += e - s
        total_clips += kept

        face = face_scores(v, ds / "faces") if v.suffix.lower() in {
            ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"} else {"error": "audio-only"}
        score, status = classify(speech_sec, kept, vol_db, face)
        print(f"  kept {kept} clips ({speech_sec:.0f}s speech) — score {score}/10 — {status}")
        report.append({
            "video": v.name, "sha1": sha1(v), "clips": kept,
            "speech_seconds": round(speech_sec, 1), "mean_volume_db": round(vol_db, 1),
            "face": face, "score": score, "status": status,
        })

    meta_csv.close()
    meta_jsonl.close()
    (out / "report.json").write_text(json.dumps({
        "clips_total": total_clips,
        "speech_seconds_total": round(sum(r["speech_seconds"] for r in report), 1),
        "face_photos_added": photos_n,
        "videos": report,
    }, indent=2, ensure_ascii=False))
    print(f"\nDONE — {total_clips} clips, "
          f"{sum(r['speech_seconds'] for r in report):.0f}s speech -> {ds}")
    print("Review dataset/metadata.csv (fix any bad transcripts) before fine-tuning.")
    for r in report:
        print(f"  {r['video']:40s} {r['score']}/10  {r['status']}")


if __name__ == "__main__":
    main()


### 3. Shrink your videos BEFORE uploading (run on your own computer)

Install ffmpeg, then for each file:

```
ffmpeg -i INPUT.mp4 -vf scale=-2:480 -c:v libx264 -crf 30 -c:a aac -b:a 96k SMALL.mp4
```

480p + compressed audio is plenty for ingestion and uploads ~10× faster.
**Even smaller:** if you don't need face crops from a given clip, upload just the
audio: `ffmpeg -i INPUT.mp4 -vn -c:a aac -b:a 96k CLIP.m4a` — then add a few
face photos in cell 4b.


In [ ]:
#@title 4. Install deps (~2-3 min)
!pip -q install faster-whisper opencv-python-headless
import os
os.makedirs('/content/videos', exist_ok=True)
os.makedirs('/content/photos', exist_ok=True)
print('done')


In [ ]:
#@title 4a. Get your files into /content/videos  (NO upload widget)
#@markdown **How:** click the **folder icon** on the left sidebar to open the
#@markdown file browser, then **drag your video/audio files into the file list**
#@markdown (drop them at the top level, next to `sample_data`). Wait for every
#@markdown upload circle to finish, **then run this cell** — it moves them into
#@markdown `videos/`. Re-drag + re-run if the runtime drops.
import os, shutil
MEDIA = ('.mp4','.mov','.mkv','.webm','.avi','.m4v','.wav','.m4a','.mp3','.aac','.ogg')
os.makedirs('/content/videos', exist_ok=True)
moved = 0
for f in list(os.listdir('/content')):
    p = f'/content/{f}'
    if os.path.isfile(p) and f.lower().endswith(MEDIA):
        shutil.move(p, f'/content/videos/{f}'); moved += 1
print(f'moved {moved} file(s). videos/ :', sorted(os.listdir('/content/videos')))


In [ ]:
#@title 4a-alt. OR pull from a Google Drive share link (no mount, server-side, fast)
#@markdown In Drive: right-click file, Share, "Anyone with the link", Copy link. Paste below.
LINK = ""  #@param {type:"string"}
NAME = "clip1.mp4"  #@param {type:"string"}
if LINK:
    !pip -q install gdown
    import gdown, re, os
    m = re.search(r'/d/([\w-]+)', LINK) or re.search(r'id=([\w-]+)', LINK)
    os.makedirs('/content/videos', exist_ok=True)
    gdown.download(id=m.group(1), output=f'/content/videos/{NAME}', quiet=False)
import os; print(sorted(os.listdir('/content/videos')))


In [ ]:
#@title 4b. (optional) FACE PHOTOS — only if you uploaded audio-only
#@markdown Drag 3-5 clear photos of your face into the file browser, then run this.
import os, shutil
os.makedirs('/content/photos', exist_ok=True)
n = 0
for f in list(os.listdir('/content')):
    p = f'/content/{f}'
    if os.path.isfile(p) and f.lower().endswith(('.jpg','.jpeg','.png','.webp')):
        shutil.move(p, f'/content/photos/{f}'); n += 1
print(f'moved {n} photo(s):', sorted(os.listdir('/content/photos')))


In [ ]:
#@title 4c. (optional) Pull a video from a direct link instead of uploading
#@markdown Use a PRIVATE / expiring link (Dropbox '?dl=1', etc.). Faster + resumable.
URL = ""  #@param {type:"string"}
NAME = "clip1.mp4"  #@param {type:"string"}
if URL:
    !wget -c -O /content/videos/$NAME "$URL" && echo OK
print(sorted(os.listdir('/content/videos')))


In [ ]:
#@title 5. Ingest — extract speech, transcribe, score
WHISPER_SIZE = "small"  #@param ["tiny", "base", "small", "medium", "large-v3"]
!python /content/ingest.py --videos /content/videos --photos /content/photos --out /content/out --whisper $WHISPER_SIZE --lang sw


In [ ]:
#@title 6. Review the result
import json, glob, random, IPython.display as ipd
rep = json.load(open('/content/out/report.json'))
print(f"TOTAL: {rep['clips_total']} clips, {rep['speech_seconds_total']:.0f}s speech, "
      f"{rep.get('face_photos_added',0)} extra photos\n")
for v in rep['videos']:
    print(f"  {v['video']:36s} {v['score']}/10  {v['status']}")
    print(f"      clips={v['clips']} speech={v['speech_seconds']}s vol={v['mean_volume_db']}dB face={v['face']}")
print('\n--- sample clips + transcripts ---')
rows = [l for l in open('/content/out/dataset/metadata.csv', encoding='utf-8')]
for line in random.sample(rows, min(4, len(rows))):
    wav, text = line.rstrip('\n').split('|', 1)
    print('\n', text); ipd.display(ipd.Audio(f'/content/out/dataset/{wav}'))
for f in sorted(glob.glob('/content/out/dataset/faces/*'))[:4]:
    print(f); ipd.display(ipd.Image(f, width=180))


### Check before moving on

- **Transcripts** right? Open `/content/out/dataset/metadata.csv` (file panel),
  fix wrong lines — bad transcripts poison a fine-tune.
- **20+ minutes** of usable speech total?
- Anything flagged `NEEDS MORE SPEECH` / `FACE NOT CLEAR ENOUGH` /
  `TOO MUCH BACKGROUND NOISE` — add more.
- Face crops recognisably you, sharp, front-ish?


In [ ]:
#@title 7. Get the dataset zip
import shutil
shutil.make_archive('/content/dataset_v1', 'zip', '/content/out')
print('made /content/dataset_v1.zip')
    from google.colab import files
    for _p in ['/content/dataset_v1.zip']:
        try:
            files.download(_p)
        except Exception as e:
            print('auto-download failed for', _p, '-', e)
    print('\nIf a download did not start: open the file browser (folder icon,'
          ' left), find dataset_v1.zip, click the 3 dots -> Download.')


## Send back

- The score/status line for each video (cell 5 or 6).
- Total usable speech seconds.
- Anything flagged.

Then keep `dataset_v1.zip` — the fine-tune notebook takes it (upload it there
the same way, or via a link).
